In [36]:
import os
import zipfile
import shutil
import pandas as pd
from pathlib import Path
from google.colab import files

In [37]:
events_step3 = pd.read_csv("events_step3_ok.csv")
reviewed = pd.read_csv("step3_flagged_files_strict_review_completed.csv")

accepted_22 = pd.read_csv("accepted_22_repaired_files.csv")
ultra_correct = pd.read_csv("ultra_correct_36_files.csv")
correct_17 = pd.read_csv("correct_17_files.csv")
correct_2 = pd.read_csv("correct_2_files.csv")

print("events_step3 rows:", len(events_step3))
print("reviewed rows:", len(reviewed))
print("accepted_22 rows:", len(accepted_22))
print("ultra_correct rows:", len(ultra_correct))
print("correct_17 rows:", len(correct_17))
print("correct_2 rows:", len(correct_2))

events_step3 rows: 2320
reviewed rows: 230
accepted_22 rows: 22
ultra_correct rows: 32
correct_17 rows: 17
correct_2 rows: 2


In [38]:
all_dfs = [events_step3, reviewed, accepted_22, ultra_correct, correct_17, correct_2]

for df in all_dfs:
    df["ticker"] = df["ticker"].astype(str).str.strip().str.upper()
    df["filing_type"] = df["filing_type"].astype(str).str.strip().str.upper()
    df["accession_number"] = df["accession_number"].astype(str).str.strip()
    df["filing_date"] = pd.to_datetime(df["filing_date"], errors="coerce").dt.strftime("%Y-%m-%d")

reviewed["manual_label"] = reviewed["manual_label"].astype(str).str.strip().str.lower()

for df in [accepted_22, ultra_correct, correct_17, correct_2]:
    if "manual_label" in df.columns:
        df["manual_label"] = df["manual_label"].astype(str).str.strip().str.lower()

/tmp/ipykernel_19666/3395013806.py:7: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df["filing_date"] = pd.to_datetime(df["filing_date"], errors="coerce").dt.strftime("%Y-%m-%d")


In [39]:
accepted_22 = accepted_22.loc[accepted_22["manual_label"] == "correct"].copy()
ultra_correct = ultra_correct.loc[ultra_correct["manual_label"] == "correct"].copy()
correct_17 = correct_17.loc[correct_17["manual_label"] == "correct"].copy()
correct_2 = correct_2.loc[correct_2["manual_label"] == "correct"].copy()

print("Accepted 22:", len(accepted_22))
print("Ultra correct:", len(ultra_correct))
print("Correct 17:", len(correct_17))
print("Correct 2:", len(correct_2))
print("Total repaired accepted rows:", len(accepted_22) + len(ultra_correct) + len(correct_17) + len(correct_2))

Accepted 22: 22
Ultra correct: 32
Correct 17: 17
Correct 2: 2
Total repaired accepted rows: 73


In [40]:
def add_row_key(df):
    df = df.copy()
    df["row_key"] = list(zip(
        df["ticker"],
        df["filing_date"],
        df["filing_type"],
        df["accession_number"]
    ))
    return df

events_step3 = add_row_key(events_step3)
reviewed = add_row_key(reviewed)
accepted_22 = add_row_key(accepted_22)
ultra_correct = add_row_key(ultra_correct)
correct_17 = add_row_key(correct_17)
correct_2 = add_row_key(correct_2)

In [41]:
flagged_all_keys = set(reviewed["row_key"])
flagged_correct_keys = set(reviewed.loc[reviewed["manual_label"] == "correct", "row_key"])
flagged_partial_keys = set(reviewed.loc[reviewed["manual_label"] == "partial", "row_key"])
flagged_incorrect_keys = set(reviewed.loc[reviewed["manual_label"] == "incorrect", "row_key"])

print("Flagged total:", len(flagged_all_keys))
print("Flagged correct:", len(flagged_correct_keys))
print("Flagged partial:", len(flagged_partial_keys))
print("Flagged incorrect:", len(flagged_incorrect_keys))

Flagged total: 230
Flagged correct: 107
Flagged partial: 33
Flagged incorrect: 90


In [42]:
events_unflagged = events_step3.loc[
    ~events_step3["row_key"].isin(flagged_all_keys)
].copy()

print("Original events_step3 rows:", len(events_step3))
print("Rows after removing all 230 flagged rows:", len(events_unflagged))

Original events_step3 rows: 2320
Rows after removing all 230 flagged rows: 2090


In [43]:
flagged_original_correct = events_step3.loc[
    events_step3["row_key"].isin(flagged_correct_keys)
].copy()

print("Flagged original rows marked correct:", len(flagged_original_correct))

Flagged original rows marked correct: 107


In [44]:
repaired_rows = pd.concat(
    [accepted_22, ultra_correct, correct_17, correct_2],
    ignore_index=True
).copy()

repaired_rows = repaired_rows.drop_duplicates(subset=["row_key"]).copy()

print("Accepted repaired rows:", len(repaired_rows))

Accepted repaired rows: 73


In [45]:
expected_final = len(events_unflagged) + len(flagged_original_correct) + len(repaired_rows)
print("Expected final row count:", expected_final)

Expected final row count: 2270


In [46]:
events_cols = events_step3.columns.tolist()

# Merge repaired row keys back to original events metadata
repaired_meta = repaired_rows[["row_key", "repaired_filename"]].merge(
    events_step3.drop_duplicates(subset=["row_key"]),
    on="row_key",
    how="left"
)

# Replace mda_path with repaired filename
repaired_meta["mda_path"] = repaired_meta["repaired_filename"]

print("Repaired metadata rows:", len(repaired_meta))
display(repaired_meta.head())

Repaired metadata rows: 73


,row_key,repaired_filename,ticker,cik,filing_date,filing_type,accession_number,year,quarter,cik_nolead,acc_nodash,mda_path,text_length_words,mda_status,primary_doc
0,"(ABT, 2019-02-22, 10-K, 0001047469-19-000624)",ABT_2019-02-22_10-K_0001047469-19-000624_REPAI...,ABT,1800,2019-02-22,10-K,0001047469-19-000624,2019,1,1800,104746919000624,ABT_2019-02-22_10-K_0001047469-19-000624_REPAI...,876,OK,a2237733z10-k.htm
1,"(ABT, 2020-02-21, 10-K, 0001104659-20-023904)",ABT_2020-02-21_10-K_0001104659-20-023904_REPAI...,ABT,1800,2020-02-21,10-K,0001104659-20-023904,2020,1,1800,110465920023904,ABT_2020-02-21_10-K_0001104659-20-023904_REPAI...,980,OK,abt-20191231x10k59d41b.htm
2,"(ABT, 2021-02-19, 10-K, 0001104659-21-025751)",ABT_2021-02-19_10-K_0001104659-21-025751_REPAI...,ABT,1800,2021-02-19,10-K,0001104659-21-025751,2021,1,1800,110465921025751,ABT_2021-02-19_10-K_0001104659-21-025751_REPAI...,907,OK,abt-20201231x10k.htm
3,"(ABT, 2022-02-18, 10-K, 0001104659-22-025141)",ABT_2022-02-18_10-K_0001104659-22-025141_REPAI...,ABT,1800,2022-02-18,10-K,0001104659-22-025141,2022,1,1800,110465922025141,ABT_2022-02-18_10-K_0001104659-22-025141_REPAI...,910,OK,abt-20211231x10k.htm
4,"(BKNG, 2019-05-09, 10-Q, 0001075531-19-000021)",BKNG_2019-05-09_10-Q_0001075531-19-000021_REPA...,BKNG,1075531,2019-05-09,10-Q,0001075531-19-000021,2019,2,1075531,107553119000021,BKNG_2019-05-09_10-Q_0001075531-19-000021_REPA...,10321,OK,bkng3311910q.htm


In [47]:
events_step3_final = pd.concat(
    [
        events_unflagged[events_cols],
        flagged_original_correct[events_cols],
        repaired_meta[events_cols]
    ],
    ignore_index=True
).copy()

print("Final rows in updated CSV:", len(events_step3_final))

Final rows in updated CSV: 2270


In [48]:
BASE_DIR = Path("/content/final_step3_package_correct")
ORIG_DIR = BASE_DIR / "orig_txt"
FINAL_DIR = BASE_DIR / "final_txt"

ACC22_DIR = BASE_DIR / "accepted22_txt"
ULTRA_DIR = BASE_DIR / "ultra_txt"
CORR17_DIR = BASE_DIR / "correct17_txt"
CORR2_DIR = BASE_DIR / "correct2_txt"

for d in [ORIG_DIR, FINAL_DIR, ACC22_DIR, ULTRA_DIR, CORR17_DIR, CORR2_DIR]:
    d.mkdir(parents=True, exist_ok=True)

FINAL_EVENTS_CSV = BASE_DIR / "events_step3_ok_updated.csv"
FINAL_ZIP = BASE_DIR / "mda_text_files_updated.zip"

In [49]:
with zipfile.ZipFile("mda_text_files.zip", "r") as zipf:
    zipf.extractall(ORIG_DIR)

with zipfile.ZipFile("accepted_22_repaired_txt.zip", "r") as zipf:
    zipf.extractall(ACC22_DIR)

with zipfile.ZipFile("ultra_correct_36_txt.zip", "r") as zipf:
    zipf.extractall(ULTRA_DIR)

with zipfile.ZipFile("correct_17_txt.zip", "r") as zipf:
    zipf.extractall(CORR17_DIR)

with zipfile.ZipFile("correct_2_txt.zip", "r") as zipf:
    zipf.extractall(CORR2_DIR)

print("Original txt files:", len(list(ORIG_DIR.glob("*.txt"))))
print("Accepted22 txt files:", len(list(ACC22_DIR.glob("*.txt"))))
print("Ultra txt files:", len(list(ULTRA_DIR.glob("*.txt"))))
print("Correct17 txt files:", len(list(CORR17_DIR.glob("*.txt"))))
print("Correct2 txt files:", len(list(CORR2_DIR.glob("*.txt"))))

Original txt files: 2380
Accepted22 txt files: 22
Ultra txt files: 32
Correct17 txt files: 17
Correct2 txt files: 2


In [68]:
with zipfile.ZipFile("mda_text_files.zip", "r") as zipf:
    zipf.extractall(ORIG_DIR)

with zipfile.ZipFile("accepted_22_repaired_txt.zip", "r") as zipf:
    zipf.extractall(ACC22_DIR)

with zipfile.ZipFile("ultra_correct_36_txt.zip", "r") as zipf:
    zipf.extractall(ULTRA_DIR)

with zipfile.ZipFile("correct_17_txt.zip", "r") as zipf:
    zipf.extractall(CORR17_DIR)

with zipfile.ZipFile("correct_2_txt.zip", "r") as zipf:
    zipf.extractall(CORR2_DIR)

print("Original txt files:", len(list(ORIG_DIR.glob("*.txt"))))
print("Accepted22 txt files:", len(list(ACC22_DIR.glob("*.txt"))))
print("Ultra txt files:", len(list(ULTRA_DIR.glob("*.txt"))))
print("Correct17 txt files:", len(list(CORR17_DIR.glob("*.txt"))))
print("Correct2 txt files:", len(list(CORR2_DIR.glob("*.txt"))))

Original txt files: 2380
Accepted22 txt files: 22
Ultra txt files: 32
Correct17 txt files: 17
Correct2 txt files: 2


In [70]:
def add_row_key(df):
    df = df.copy()
    df["ticker"] = df["ticker"].astype(str).str.strip().str.upper()
    df["filing_type"] = df["filing_type"].astype(str).str.strip().str.upper()
    df["accession_number"] = df["accession_number"].astype(str).str.strip()
    df["filing_date"] = pd.to_datetime(df["filing_date"], errors="coerce").dt.strftime("%Y-%m-%d")

    df["row_key"] = list(zip(
        df["ticker"],
        df["filing_date"],
        df["filing_type"],
        df["accession_number"]
    ))
    return df

accepted_22 = add_row_key(accepted_22)
ultra_correct = add_row_key(ultra_correct)
correct_17 = add_row_key(correct_17)
correct_2 = add_row_key(correct_2)

print("accepted_22 has row_key:", "row_key" in accepted_22.columns)
print("ultra_correct has row_key:", "row_key" in ultra_correct.columns)
print("correct_17 has row_key:", "row_key" in correct_17.columns)
print("correct_2 has row_key:", "row_key" in correct_2.columns)

accepted_22 has row_key: True
ultra_correct has row_key: True
correct_17 has row_key: True
correct_2 has row_key: True


In [71]:
def build_filename_map(df):
    return {
        r["row_key"]: r["repaired_filename"]
        for _, r in df.iterrows()
    }

accepted_22_map = build_filename_map(accepted_22)
ultra_map = build_filename_map(ultra_correct)
correct_17_map = build_filename_map(correct_17)
correct_2_map = build_filename_map(correct_2)

In [72]:
for old_file in FINAL_DIR.glob("*.txt"):
    old_file.unlink()

In [73]:
copied_unflagged = 0
missing_unflagged = []

for _, row in events_unflagged.iterrows():
    orig_name = os.path.basename(str(row["mda_path"]))
    src = ORIG_DIR / orig_name
    dst = FINAL_DIR / orig_name

    if src.exists():
        shutil.copy2(src, dst)
        copied_unflagged += 1
    else:
        missing_unflagged.append((row["row_key"], str(src)))

print("Copied unflagged original files:", copied_unflagged)
print("Missing unflagged original files:", len(missing_unflagged))

Copied unflagged original files: 2085
Missing unflagged original files: 5


In [74]:
copied_flagged_correct = 0
missing_flagged_correct = []

for _, row in flagged_original_correct.iterrows():
    orig_name = os.path.basename(str(row["mda_path"]))
    src = ORIG_DIR / orig_name
    dst = FINAL_DIR / orig_name

    if src.exists():
        shutil.copy2(src, dst)
        copied_flagged_correct += 1
    else:
        missing_flagged_correct.append((row["row_key"], str(src)))

print("Copied flagged correct original files:", copied_flagged_correct)
print("Missing flagged correct original files:", len(missing_flagged_correct))

Copied flagged correct original files: 107
Missing flagged correct original files: 0


In [75]:
copied_repaired = 0
missing_repaired = []

for _, row in repaired_rows.iterrows():
    key = row["row_key"]

    if key in correct_2_map:
        fname = correct_2_map[key]
        src = CORR2_DIR / fname
    elif key in correct_17_map:
        fname = correct_17_map[key]
        src = CORR17_DIR / fname
    elif key in ultra_map:
        fname = ultra_map[key]
        src = ULTRA_DIR / fname
    elif key in accepted_22_map:
        fname = accepted_22_map[key]
        src = ACC22_DIR / fname
    else:
        continue

    dst = FINAL_DIR / fname

    if src.exists():
        shutil.copy2(src, dst)
        copied_repaired += 1
    else:
        missing_repaired.append((key, str(src)))

print("Copied repaired files:", copied_repaired)
print("Missing repaired files:", len(missing_repaired))

Copied repaired files: 73
Missing repaired files: 0


In [76]:
repaired_key_to_fname = {}
repaired_key_to_fname.update(accepted_22_map)
repaired_key_to_fname.update(ultra_map)
repaired_key_to_fname.update(correct_17_map)
repaired_key_to_fname.update(correct_2_map)

events_step3_final = add_row_key(events_step3_final)

events_step3_final["mda_path"] = events_step3_final.apply(
    lambda r: repaired_key_to_fname.get(r["row_key"], os.path.basename(str(r["mda_path"]))),
    axis=1
)

events_step3_final = events_step3_final.drop(columns=["row_key"])
print("Final CSV rows:", len(events_step3_final))

Final CSV rows: 2270


In [77]:
events_step3_final.to_csv(FINAL_EVENTS_CSV, index=False)
print("Saved:", FINAL_EVENTS_CSV)

Saved: /content/final_step3_package_correct/events_step3_ok_updated.csv


In [78]:
with zipfile.ZipFile(FINAL_ZIP, "w", zipfile.ZIP_DEFLATED) as zipf:
    for txt_file in FINAL_DIR.glob("*.txt"):
        zipf.write(txt_file, arcname=txt_file.name)

print("Saved:", FINAL_ZIP)

Saved: /content/final_zip_build/mda_text_files_updated.zip


In [79]:
with zipfile.ZipFile(FINAL_ZIP, "r") as zipf:
    zip_names = set(zipf.namelist())

csv_names = set(events_step3_final["mda_path"].astype(str))

print("Final CSV rows:", len(events_step3_final))
print("Final ZIP files:", len(zip_names))
print("Missing in ZIP:", len(csv_names - zip_names))
print("Extra in ZIP:", len(zip_names - csv_names))

Final CSV rows: 2270
Final ZIP files: 2265
Missing in ZIP: 5
Extra in ZIP: 0


In [80]:
with zipfile.ZipFile(FINAL_ZIP, "r") as zipf:
    zip_names = set(zipf.namelist())

csv_names = set(events_step3_final["mda_path"].astype(str))

missing_files = sorted(csv_names - zip_names)

print("Missing files count:", len(missing_files))
for x in missing_files:
    print(x)

Missing files count: 5
CSCO_20220222_10-Q_000085887722000004.txt
DUK_20241107_10-Q_000132616024000181.txt
KO_20200722_10-Q_000002134420000041.txt
NKE_20211005_10-Q_000032018721000045.txt
STX_20220428_10-Q_000113778922000031.txt


In [81]:
missing_rows = events_step3_final.loc[
    events_step3_final["mda_path"].astype(str).isin(missing_files)
].copy()

print("Rows with missing files:", len(missing_rows))
display(missing_rows[["ticker", "filing_date", "filing_type", "accession_number", "mda_path"]])

Rows with missing files: 5


,ticker,filing_date,filing_type,accession_number,mda_path
569,CSCO,2022-02-22,10-Q,0000858877-22-000004,CSCO_20220222_10-Q_000085887722000004.txt
699,DUK,2024-11-07,10-Q,0001326160-24-000181,DUK_20241107_10-Q_000132616024000181.txt
1042,KO,2020-07-22,10-Q,0000021344-20-000041,KO_20200722_10-Q_000002134420000041.txt
1382,NKE,2021-10-05,10-Q,0000320187-21-000045,NKE_20211005_10-Q_000032018721000045.txt
1764,STX,2022-04-28,10-Q,0001137789-22-000031,STX_20220428_10-Q_000113778922000031.txt


In [82]:
search_dirs = [ORIG_DIR, ACC22_DIR, ULTRA_DIR, CORR17_DIR, CORR2_DIR]

for fname in missing_files:
    found = []
    for d in search_dirs:
        p = d / fname
        if p.exists():
            found.append(str(p))
    print("\nFILE:", fname)
    if found:
        print("FOUND IN:")
        for f in found:
            print("  ", f)
    else:
        print("NOT FOUND ANYWHERE")


FILE: CSCO_20220222_10-Q_000085887722000004.txt
NOT FOUND ANYWHERE

FILE: DUK_20241107_10-Q_000132616024000181.txt
NOT FOUND ANYWHERE

FILE: KO_20200722_10-Q_000002134420000041.txt
NOT FOUND ANYWHERE

FILE: NKE_20211005_10-Q_000032018721000045.txt
NOT FOUND ANYWHERE

FILE: STX_20220428_10-Q_000113778922000031.txt
NOT FOUND ANYWHERE


In [83]:
events_step3_final = events_step3_final.loc[
    ~events_step3_final["mda_path"].astype(str).isin(missing_files)
].copy()

print("Final CSV rows after dropping true-missing files:", len(events_step3_final))

Final CSV rows after dropping true-missing files: 2265


In [84]:
events_step3_final.to_csv(FINAL_EVENTS_CSV, index=False)

with zipfile.ZipFile(FINAL_ZIP, "w", zipfile.ZIP_DEFLATED) as zipf:
    for txt_file in FINAL_DIR.glob("*.txt"):
        zipf.write(txt_file, arcname=txt_file.name)

with zipfile.ZipFile(FINAL_ZIP, "r") as zipf:
    zip_names = set(zipf.namelist())

csv_names = set(events_step3_final["mda_path"].astype(str))

print("Final CSV rows:", len(events_step3_final))
print("Final ZIP files:", len(zip_names))
print("Missing in ZIP:", len(csv_names - zip_names))
print("Extra in ZIP:", len(zip_names - csv_names))

Final CSV rows: 2265
Final ZIP files: 2265
Missing in ZIP: 0
Extra in ZIP: 0


In [85]:
from google.colab import files

files.download(str(FINAL_EVENTS_CSV))
files.download(str(FINAL_ZIP))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>